# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from pathlib import Path
from src.lab_day10_solution import inspect_splits, visualize_split_examples

summary = inspect_splits('data/splits')
summary


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from src.lab_day10_solution import (
    PROMPT_NORM1000, answer_to_pixel_boxes, objects_to_answer, yolo_to_box
)
from scripts.labcommon import read_jsonl

rows = read_jsonl('data/splits/train.jsonl')
example = rows[0]
answer = objects_to_answer(example['objects'], width=example['width'], height=example['height'])
print(PROMPT_NORM1000)
print(answer)
print(answer_to_pixel_boxes(answer, example['width'], example['height']))


## 3. E0: zero-shot

In [ ]:
from src.lab_day10_solution import PROMPT_NORM1000, load_qwen, predict_split

model, processor = load_qwen('Qwen/Qwen3-VL-4B-Instruct')
predict_split(
    model, processor, split='test', out_path='work/e0.jsonl',
    data_dir='data', prompt=PROMPT_NORM1000, batch_size=4, max_new_tokens=256
)


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from src.lab_day10_solution import RoadDamageDataset, encode_example, make_collate

train_ds = RoadDamageDataset('train', data_dir='data')
encoded = encode_example(processor, train_ds[0]['image'], train_ds[0]['prompt'], train_ds[0]['answer'])
label_tokens = encoded['input_ids'][encoded['labels'] != -100]
print('answer tokens:', processor.tokenizer.decode(label_tokens, skip_special_tokens=False))
if 'image_grid_thw' in encoded:
    print('image_grid_thw:', encoded['image_grid_thw'])


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from pathlib import Path
import yaml
from src.lab_day10_solution import apply_setup

# Example only. For each experiment, load a fresh base model, then apply its config.
setup = yaml.safe_load(Path('configs/e1.yaml').read_text(encoding='utf-8'))
# model = apply_setup(model, setup)
setup


## 6. Vòng huấn luyện

In [ ]:
from src.lab_day10_solution import train_experiment

# Run one at a time on Colab T4. Each command trains and saves work/adapters/eN/.
# run_e1 = train_experiment('configs/e1.yaml')
# run_e2 = train_experiment('configs/e2.yaml')
# run_e3 = train_experiment('configs/e3.yaml')


## 7. Suy luận và chấm E1–E3

In [ ]:
from src.lab_day10_solution import infer_adapter, score_experiment

# Run after each training job finishes.
# infer_adapter('configs/e1.yaml'); score_experiment('configs/e1.yaml')
# infer_adapter('configs/e2.yaml'); score_experiment('configs/e2.yaml')
# infer_adapter('configs/e3.yaml'); score_experiment('configs/e3.yaml')


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
from src.lab_day10_solution import run_reward_tests, train_experiment, infer_adapter, score_experiment

# E4: choose one clean ablation in configs/e4.yaml, then train/infer/score it.
# run_e4 = train_experiment('configs/e4.yaml')
# infer_adapter('configs/e4.yaml'); score_experiment('configs/e4.yaml')

# E5 reward unit tests, no GPU needed.
run_reward_tests()
print('reward tests passed')
